**Credit Risk Modelling**

import libaries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import seaborn as sns

read data

In [ ]:
df = pd.read_csv("BankChurners.csv") #convert csv file to readable datafram

In [ ]:
#first five columns of data
df.head()

Identify Null Values in data

In [ ]:
if df.isnull().any(axis=1).sum() == 0:
    print("No null values")
else:
    print(df[df.isnull().any(axis=1)])

Data transformation

In [ ]:
#since CLIENTNUM is just an identifier, we will that column as our index
df = df.set_index(df["CLIENTNUM"])
df = df.drop(columns=["CLIENTNUM"])


In [ ]:
df.head()

In [ ]:
#The last two columns will not be used as they are displaying the probability of a customer being an existing customer or attrited customer
df = df.iloc[:, :-2]

Identify outliers or odd values (error in data entry)

In [ ]:
cols = df.select_dtypes(include=["int64","float64"]).columns

col1 = []
col2 = []
for x in range(0,len(cols)//2):
    col1.append(cols[x])
for y in range(len(cols)//2, len(cols)):
    col2.append(cols[y])

In [ ]:

fig, axes = plt.subplots(1, len(col1), figsize=(12, 4))
for ax, c in zip(axes, col1):
    ax.boxplot(df[c])
    ax.set_title(c)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(col2), figsize=(12, 4))
for ax, c in zip(axes, col2):
    ax.boxplot(df[c])
    ax.set_title(c)
plt.tight_layout()
plt.show()

Since there isnt negative values or values that might be impossible we'll assume there is no error in data entry

**Turn all data types to numeric**

In [ ]:
columns = df.select_dtypes(include="object").columns
for x in columns:
    print(f"{x}:{df[x].unique()}")

In [ ]:
#We cannot use get_dummies in this scenario as there would be too much columns for a proper EDA
#Given that I have planned to use Random Forest Classifier to predict the final outcome, we will label each value as numeric values e.g (0 - 10), it would also be appropriate since we are
#able to see a clear ordinal value 

df["Attrition_Flag"] = df["Attrition_Flag"].map({'Existing Customer':0, 'Attrited Customer':1})
df["Gender"] = df["Gender"].map({'M':1, 'F':0})

mappings = {

    "Education_Level" : {
        "Unknown":0,
        "Uneducated":1,
        "High School":2,
        "College":3,
        "Graduate":4,
        "Post-Graduate":5,
        "Doctorate":6
    },

    "Marital_Status" : {
        "Unknown":0,
        "Single":1,
        "Married":2,
        "Divorced":3
    },

    "Income_Category" : {
        "Unknown":0,
        'Less than $40K':1,
        "$40K - $60K":2,
        "$60K - $80K":3,
        "$80K - $120K":4,
        "$120K +":5
    },

    "Card_Category":
    {
        "Blue":0,
        "Silver":1,
        "Gold":2,
        "Platinum":3
    }
}


for col, mapping in mappings.items():
    df[col] = df[col].map(mapping)


In [ ]:
for x in df.columns:
    print(x,df[x].max())

In [ ]:
df.head()

In [ ]:
#Final Check
df.dtypes

In [ ]:
df_raw = df

In [ ]:
df_raw.head()

**EDA**

pearson value and r value

In [ ]:
target = 'Attrition_Flag'
for col in df.columns:
    if col == target:
        continue
    r, p = pearsonr(df[col],df[target])
    print(f"{col:25s} r={r:+.3f} p={p:.4f}")

What can be deduced from this:
REMINDER: O = existing, 1 = attirtion 
Looking at purely r-value:

RELEVANT
Total_Trans_Ct            r=-0.371 p=0.0000 (total number of transactions over 12 months.)
Total_Ct_Chng_Q4_Q1       r=-0.290 p=0.0000 (difference in number of transactions from Q4 to Q1)
Total_Revolving_Bal       r=-0.263 p=0.0000 (total amount of debt held month to month)
Contacts_Count_12_mon     r=+0.204 p=0.0000 (number of times person has contacted the bank)
Avg_Utilization_Ratio     r=-0.178 p=0.0000 (how much of credit customer uses over total credit)
Total_Trans_Amt           r=-0.169 p=0.0000 (how many dollars are used)
Months_Inactive_12_mon    r=+0.152 p=0.0000 (months with no activity in the last year.)

LEFT OUT:
Be further reconsidered
Customer_Age              r=+0.018 p=0.0670 
Gender                    r=-0.037 p=0.0002
Dependent_count           r=+0.019 p=0.0560
Education_Level           r=+0.009 p=0.3761
Marital_Status            r=-0.019 p=0.0613
Income_Category           r=-0.014 p=0.1719
Card_Category             r=+0.002 p=0.8128
Months_on_book            r=+0.014 p=0.1684
Avg_Open_To_Buy           r=-0.000 p=0.9771 (confirmed left out)
Credit_Limit              r=-0.024 p=0.0163


Reconsider the left out columns

Customer Age

In [ ]:
#customer age
df["customer_age_category"] = pd.qcut(df["Customer_Age"],q=4)
print(df.groupby(df["customer_age_category"], observed=False)["Attrition_Flag"].mean())
df = df.drop(columns=["customer_age_category"])

the rate is close for all age groups meaning there is genuinely no relation to age and whether a customer is still existing or attrition

Customer months on book

In [ ]:
df["months_on_book_cat"] = pd.qcut(df["Months_on_book"],q=4)
print(df.groupby(df["months_on_book_cat"], observed=False)["Attrition_Flag"].mean())
df= df.drop(columns=["months_on_book_cat"])

In [ ]:
df["Credit_Limit_cat"] = pd.qcut(df["Credit_Limit"],q=4)
print(df.groupby(df["Credit_Limit_cat"], observed=False)["Attrition_Flag"].mean())
df= df.drop(columns=["Credit_Limit_cat"])

Genuine find: a customer is more likely to churn if it's given a low credit_limit, customer might switch to other banks that can easily give more

Customer Gender(and other categorical values)

In [ ]:
values = ["Gender","Marital_Status","Education_Level","Card_Category","Dependent_count"]

for x in values:
    print(df.groupby(df[x], observed=False)["Attrition_Flag"].mean())
    print()

These columns do not show any meaningful correlation to whether a person is an existing customer or not, which supports what our r value has stated
Columns such as card_category and education_level have a increasing value because the counts of each category in the column gets lesser, so it would a person who is has 1 for attrition flag will have significant change in value of mean

**Conclusion from EDA**

all columns that were left out can be genuinely dropped except for credt limit

In [ ]:
cols_left_out = [
    "Customer_Age",
    "Gender",
    "Dependent_count",
    "Education_Level",
    "Marital_Status",
    "Income_Category",
    "Card_Category",
    "Months_on_book",
    "Avg_Open_To_Buy",
]

In [ ]:
df = df.drop(columns=cols_left_out)

In [ ]:
df.head()

**Split the dataset by 80/20**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_clean = df.drop(columns="Attrition_Flag")
Y_clean = df["Attrition_Flag"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X_clean, Y_clean, test_size = 0.2, random_state=42, stratify=Y_clean
)


**RandomForestClassifier model fitting**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model_clean= RandomForestClassifier(random_state=42,class_weight="balanced")
model_clean.fit(X_train,Y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model_clean.predict(X_test)

In [ ]:
print(confusion_matrix(Y_test,y_pred))

In [ ]:
print(classification_report(Y_test,y_pred))

Prediction result without dropping the left out columns (excluding the naiva bayes columns)

In [ ]:
df_raw = df_raw.drop(columns="customer_age_category")

In [ ]:
X_raw = df_raw.drop(columns=("Attrition_Flag"))
Y_raw = df_raw["Attrition_Flag"]

In [ ]:
X_raw.head()

In [ ]:
X_raw_train, X_raw_test, Y_raw_train, Y_raw_test = train_test_split( X_raw, Y_raw, test_size=0.2, random_state=43, stratify=Y_raw)

In [ ]:
model_raw = RandomForestClassifier(random_state=43, class_weight="balanced")
model_raw.fit(X_raw_train,Y_raw_train)
y_raw_pred = model_raw.predict(X_raw_test)


In [ ]:
print(confusion_matrix(Y_raw_test,y_raw_pred))

In [ ]:
print(classification_report(Y_raw_test,y_raw_pred))

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
for metric in ['recall', 'precision', 'f1', 'roc_auc', 'average_precision']:
    clean = cross_val_score(model_clean, X_clean, Y_clean, cv=5, scoring=metric).mean()
    raw   = cross_val_score(model_raw,   X_raw,   Y_raw,   cv=5, scoring=metric).mean()
    print(f"{metric:18s}  clean={clean:.3f}  raw={raw:.3f}")